# ADJSCC-CSINet+ Tuned k=32

This is the main notebook. It targets the paper-style `k=32` regime, keeps the model compatible with signed normalized CSI, and adds the higher-upside training fixes intended to break the current `~-3.5 dB` validation plateau.


## What changed

- Uses one consistent config source for loader, model, and training loop.
- Computes validation/test NMSE over the full split in linear domain and converts to dB once.
- Prints feedback bandwidth from the dataset config and the chosen `k`.
- Uses deterministic validation/test traversal and checkpointing by linear NMSE.


In [ ]:
import math
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
@dataclass
class TrainingConfig:
    train_file: str = "train_data.mat"
    val_file: str = "val_data.mat"
    test_file: str = "test_data.mat"
    checkpoint_dir: str = "checkpoints_tuned_k32"
    fast_dev_run: bool = False
    run_training: bool = False
    epochs: int = 500
    batch_size: int = 200
    learning_rate: float = 0.001
    min_lr: float = 0.0001
    patience: int = 20
    weight_decay: float = 1e-05
    grad_clip: float = 1.0
    snr_low: float = -10.0
    snr_high: float = 10.0
    k_feedback: int = 32
    compression_ratio: int = 32
    mse_weight: float = 0.5
    nmse_weight: float = 0.5
    warmup_epochs: int = 20
    save_every: int = 10
    run_evaluation: bool = False
    checkpoint_path: str | None = None
    resume_latest: bool = False


cfg = TrainingConfig()
os.makedirs(cfg.checkpoint_dir, exist_ok=True)

try:
    script_args
except NameError:
    script_args = None

if script_args is not None:
    cfg.run_training = bool(script_args.train)
    cfg.run_evaluation = bool(
        script_args.evaluate or script_args.train or script_args.checkpoint or script_args.resume_latest
    )
    cfg.fast_dev_run = bool(script_args.fast_dev_run)
    cfg.checkpoint_path = script_args.checkpoint
    cfg.resume_latest = bool(script_args.resume_latest)
    if script_args.epochs is not None:
        cfg.epochs = int(script_args.epochs)
    if script_args.batch_size is not None:
        cfg.batch_size = int(script_args.batch_size)
else:
    cfg.run_evaluation = False

if cfg.resume_latest and cfg.checkpoint_path is not None:
    raise ValueError("Use either --checkpoint or --resume-latest, not both.")


def _read_scalar(dataset):
    value = dataset[()]
    if isinstance(value, np.ndarray) and value.size == 1:
        return float(value.reshape(-1)[0])
    return value


def load_dataset_cfg(path: str):
    with h5py.File(path, "r") as f:
        group = f["cfg"]
        out = {key: _read_scalar(group[key]) for key in group.keys() if isinstance(group[key], h5py.Dataset)}
    return out


dataset_cfg = load_dataset_cfg(cfg.train_file)
num_subcarriers = int(dataset_cfg["num_subcarriers"])
subcarrier_bw_hz = float(dataset_cfg["bandwidth"]) / num_subcarriers
feedback_bw_hz = cfg.k_feedback * subcarrier_bw_hz

print(f"Configured feedback symbols k = {cfg.k_feedback}")
print(f"Compression ratio = {cfg.compression_ratio}")
print(f"Feedback bandwidth = {feedback_bw_hz / 1e6:.3f} MHz ({int(round(feedback_bw_hz))} Hz)")
print(f"Subcarrier spacing = {subcarrier_bw_hz / 1e3:.3f} kHz")


def get_train_global_scale(train_path, chunk_size=500):
    stats = {"dl": {"sum_sq": 0.0, "count": 0}, "ul": {"sum_sq": 0.0, "count": 0}}
    with h5py.File(train_path, "r") as f:
        for key, short_name in [("csi_dl", "dl"), ("csi_ul", "ul")]:
            dataset = f[key]
            num_samples = dataset.shape[3]
            for start in range(0, num_samples, chunk_size):
                stop = min(start + chunk_size, num_samples)
                chunk = dataset[:, :, :, start:stop]
                real = chunk["real"].astype(np.float32)
                imag = chunk["imag"].astype(np.float32)
                stats[short_name]["sum_sq"] += float(np.sum(real ** 2) + np.sum(imag ** 2))
                stats[short_name]["count"] += real.size + imag.size

    out = {}
    for key in ("dl", "ul"):
        var = stats[key]["sum_sq"] / max(stats[key]["count"], 1)
        out[key] = {"std": float(np.sqrt(var + 1e-12))}
    print("Train-only scale stats:", out)
    return out


class CSIDatasetManager:
    def __init__(self, train_path, val_path, test_path, stats):
        self.stats = stats
        self.paths = {"train": train_path, "val": val_path, "test": test_path}
        self.files = {}
        self.datasets = {}
        self.lengths = {}

        for split, path in self.paths.items():
            handle = h5py.File(path, "r")
            self.files[split] = handle
            self.datasets[split] = {"dl": handle["csi_dl"], "ul": handle["csi_ul"]}
            self.lengths[split] = int(handle["csi_dl"].shape[3])
            print(f"{split} samples: {self.lengths[split]}")

    def close(self):
        for handle in self.files.values():
            handle.close()

    def _normalize(self, arr, key):
        return arr / (self.stats[key]["std"] + 1e-8)

    def denormalize(self, tensor, key):
        return tensor * (self.stats[key]["std"] + 1e-8)

    def _process(self, batch_arr, key, normalize=True):
        real = batch_arr["real"].astype(np.float32)
        imag = batch_arr["imag"].astype(np.float32)
        if normalize:
            real = self._normalize(real, key)
            imag = self._normalize(imag, key)
        merged = np.stack([real, imag], axis=2)
        merged = np.squeeze(merged, axis=3)
        merged = np.transpose(merged, (3, 2, 0, 1))
        return merged

    def get_batch(self, split, indices, snr_values=None):
        indices = np.asarray(indices, dtype=np.int64)
        indices.sort()
        dl_raw = self.datasets[split]["dl"][:, :, :, indices]
        ul_raw = self.datasets[split]["ul"][:, :, :, indices]

        dl = torch.from_numpy(self._process(dl_raw, "dl", normalize=True)).float()
        ul = torch.from_numpy(self._process(ul_raw, "ul", normalize=True)).float()

        if snr_values is None:
            snr_values = np.random.uniform(cfg.snr_low, cfg.snr_high, size=(len(indices), 1)).astype(np.float32)
        else:
            snr_values = np.asarray(snr_values, dtype=np.float32).reshape(len(indices), 1)

        snr = torch.from_numpy(snr_values).float()
        return dl, ul, snr

    def iterate_split(self, split, batch_size, shuffle=False, generator=None, fixed_snr=None):
        total = self.lengths[split]
        order = np.arange(total, dtype=np.int64)
        if shuffle:
            rng = generator if generator is not None else np.random.default_rng()
            rng.shuffle(order)

        for start in range(0, total, batch_size):
            batch_indices = order[start:start + batch_size]
            snr_values = None
            if fixed_snr is not None:
                snr_values = np.full((len(batch_indices), 1), fixed_snr, dtype=np.float32)
            yield self.get_batch(split, batch_indices, snr_values=snr_values)


stats = get_train_global_scale(cfg.train_file)
dataset = CSIDatasetManager(cfg.train_file, cfg.val_file, cfg.test_file, stats)

if cfg.fast_dev_run:
    debug_train_batches = 8
    debug_eval_batches = 4
else:
    debug_train_batches = None
    debug_eval_batches = None


In [ ]:
class AFModule(nn.Module):
    def __init__(self, channels, reduction_ratio=2):
        super().__init__()
        hidden_dim = max(channels // reduction_ratio, 1)
        self.fc1 = nn.Linear(channels + 1, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, channels)

    def forward(self, x, snr):
        pooled = F.adaptive_avg_pool2d(x, 1).flatten(1)
        scale = torch.cat([pooled, snr], dim=1)
        scale = F.relu(self.fc1(scale))
        scale = torch.sigmoid(self.fc2(scale)).view(x.size(0), x.size(1), 1, 1)
        return x * scale


class ATN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 16, kernel_size=3, stride=(2, 1), padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.prelu1 = nn.PReLU()
        self.af1 = AFModule(16)

        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, stride=(2, 1), padding=1)
        self.bn2 = nn.BatchNorm2d(16)
        self.prelu2 = nn.PReLU()
        self.af2 = AFModule(16)

        self.conv3 = nn.Conv2d(16, 2, kernel_size=3, stride=(2, 1), padding=1)
        self.bn3 = nn.BatchNorm2d(2)

    def forward(self, x, snr):
        x = self.af1(self.prelu1(self.bn1(self.conv1(x))), snr)
        x = self.af2(self.prelu2(self.bn2(self.conv2(x))), snr)
        x = self.bn3(self.conv3(x))
        return x


class CsiNetPlusEncoderWithAF(nn.Module):
    def __init__(self, compression_ratio):
        super().__init__()
        self.input_channels = 2
        self.height = 32
        self.width = 32
        self.total_elements = self.input_channels * self.height * self.width
        self.M = self.total_elements // compression_ratio

        self.conv1 = nn.Conv2d(2, 2, kernel_size=7, stride=1, padding=3)
        self.bn1 = nn.BatchNorm2d(2)
        self.af1 = AFModule(2)

        self.conv2 = nn.Conv2d(2, 2, kernel_size=7, stride=1, padding=3)
        self.bn2 = nn.BatchNorm2d(2)
        self.af2 = AFModule(2)

        self.fc = nn.Linear(self.total_elements, self.M)

    def forward(self, x, snr):
        x = self.af1(F.leaky_relu(self.bn1(self.conv1(x)), negative_slope=0.3), snr)
        x = self.af2(F.leaky_relu(self.bn2(self.conv2(x)), negative_slope=0.3), snr)
        x = x.flatten(1)
        return self.fc(x)


def enc_to_complex_and_normalize(encoder_output):
    k = encoder_output.shape[1] // 2
    real_part = encoder_output[:, :k]
    imag_part = encoder_output[:, k:]
    s = torch.complex(real_part, imag_part)
    power = torch.mean(s.abs().square(), dim=1, keepdim=True)
    return s / torch.sqrt(power + 1e-8)


class WirelessChannelSimulator(nn.Module):
    def __init__(self, num_bs_antennas=32, training_random_subcarriers=True):
        super().__init__()
        self.Nt = num_bs_antennas
        self.training_random_subcarriers = training_random_subcarriers

    def _select_indices(self, num_subcarriers, k, device):
        if self.training and self.training_random_subcarriers:
            return torch.randperm(num_subcarriers, device=device)[:k]
        return torch.linspace(0, num_subcarriers - 1, steps=k, device=device).round().long()

    def forward(self, s, snr_db, h_uplink_raw):
        batch_size, k = s.shape
        device = s.device

        num_subcarriers = h_uplink_raw.shape[2]
        if num_subcarriers < k:
            raise ValueError(f"Need at least {k} subcarriers, found {num_subcarriers}")

        indices = self._select_indices(num_subcarriers, k, device)
        h_sliced = h_uplink_raw[:, :, indices, :]
        h_u = torch.complex(h_sliced[:, 0], h_sliced[:, 1])

        snr_linear = torch.pow(10.0, snr_db / 10.0)
        noise_power = 1.0 / snr_linear
        noise_std = torch.sqrt(noise_power / 2.0).unsqueeze(-1)

        z_real = torch.randn(batch_size, k, self.Nt, device=device) * noise_std
        z_imag = torch.randn(batch_size, k, self.Nt, device=device) * noise_std
        z = torch.complex(z_real, z_imag)

        y = h_u * s.unsqueeze(-1) + z
        w = h_u / (torch.norm(h_u, dim=2, keepdim=True) + 1e-8)
        return torch.sum(torch.conj(w) * y, dim=2)


class ComplexToReal(nn.Module):
    def forward(self, s_hat):
        return torch.cat([s_hat.real, s_hat.imag], dim=1)


class ModifiedRefineNetBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, 8, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm2d(8)
        self.af1 = AFModule(8)

        self.conv2 = nn.Conv2d(8, 16, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm2d(16)
        self.af2 = AFModule(16)

        self.conv3 = nn.Conv2d(16, channels, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(channels)
        self.af3 = AFModule(channels)

    def forward(self, x, snr):
        residual = x
        x = self.af1(F.leaky_relu(self.bn1(self.conv1(x)), negative_slope=0.3), snr)
        x = self.af2(F.leaky_relu(self.bn2(self.conv2(x)), negative_slope=0.3), snr)
        x = self.af3(self.bn3(self.conv3(x)), snr)
        return residual + x


class CsiNetPlusDecoder(nn.Module):
    def __init__(self, input_dim, height=32, width=32, channels=2, num_blocks=5):
        super().__init__()
        self.height = height
        self.width = width
        self.channels = channels
        self.flattened_dim = height * width * channels

        self.fc = nn.Linear(input_dim, self.flattened_dim)
        self.initial_conv = nn.Conv2d(channels, channels, kernel_size=7, padding=3)
        self.initial_bn = nn.BatchNorm2d(channels)
        self.initial_af = AFModule(channels)
        self.refinenet_chain = nn.ModuleList([ModifiedRefineNetBlock(channels) for _ in range(num_blocks)])

    def forward(self, x, snr):
        x = self.fc(x).view(-1, self.channels, self.height, self.width)
        x = self.initial_af(F.leaky_relu(self.initial_bn(self.initial_conv(x)), negative_slope=0.3), snr)
        for block in self.refinenet_chain:
            x = block(x, snr)
        return x


class STN(nn.Module):
    def __init__(self):
        super().__init__()
        self.trans_conv1 = nn.ConvTranspose2d(2, 16, kernel_size=3, stride=(2, 1), padding=1, output_padding=(1, 0))
        self.bn1 = nn.BatchNorm2d(16)
        self.prelu1 = nn.PReLU()
        self.af1 = AFModule(16)

        self.trans_conv2 = nn.ConvTranspose2d(16, 16, kernel_size=3, stride=(2, 1), padding=1, output_padding=(1, 0))
        self.bn2 = nn.BatchNorm2d(16)
        self.prelu2 = nn.PReLU()
        self.af2 = AFModule(16)

        self.trans_conv3 = nn.ConvTranspose2d(16, 2, kernel_size=3, stride=(2, 1), padding=1, output_padding=(1, 0))
        self.bn3 = nn.BatchNorm2d(2)

    def forward(self, x, snr):
        x = self.af1(self.prelu1(self.bn1(self.trans_conv1(x))), snr)
        x = self.af2(self.prelu2(self.bn2(self.trans_conv2(x))), snr)
        x = self.bn3(self.trans_conv3(x))
        return x


atn = ATN().to(device)
encoder = CsiNetPlusEncoderWithAF(compression_ratio=cfg.compression_ratio).to(device)
channel_sim = WirelessChannelSimulator(num_bs_antennas=int(dataset_cfg["num_bs_antennas"])).to(device)
c2r = ComplexToReal().to(device)
decoder = CsiNetPlusDecoder(input_dim=encoder.M).to(device)
stn = STN().to(device)

all_params = list(atn.parameters()) + list(encoder.parameters()) + list(decoder.parameters()) + list(stn.parameters())
total_params = sum(p.numel() for p in all_params if p.requires_grad)

print(f"Encoder output dimension M = {encoder.M}")
print(f"Feedback symbols k = {encoder.M // 2}")
print(f"Total trainable parameters = {total_params:,}")


In [ ]:
def samplewise_linear_nmse(h_true, h_pred):
    numerator = torch.sum((h_true - h_pred) ** 2, dim=(1, 2, 3))
    denominator = torch.sum(h_true ** 2, dim=(1, 2, 3)) + 1e-8
    return numerator / denominator


def nmse_db_from_sums(error_sum, power_sum):
    return 10.0 * math.log10((error_sum / max(power_sum, 1e-12)) + 1e-12)


optimizer = optim.Adam(all_params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=cfg.patience,
    min_lr=cfg.min_lr,
)
mse_criterion = nn.MSELoss()


def forward_pass(H_d, H_u, snr):
    T = atn(H_d, snr)
    c = encoder(T, snr)
    s = enc_to_complex_and_normalize(c)
    s_hat = channel_sim(s, snr, H_u)
    c_hat = c2r(s_hat)
    T_hat = decoder(c_hat, snr)
    H_hat = stn(T_hat, snr)
    return H_hat


def compute_loss(H_true, H_pred, epoch_index):
    mse_loss = mse_criterion(H_pred, H_true)
    nmse_linear = samplewise_linear_nmse(H_true, H_pred).mean()
    if epoch_index < cfg.warmup_epochs:
        total_loss = mse_loss
    else:
        total_loss = cfg.mse_weight * mse_loss + cfg.nmse_weight * nmse_linear
    return total_loss, mse_loss.detach(), nmse_linear.detach()


def run_epoch(split, epoch_index=0, fixed_snr=None):
    is_train = split == "train"
    models = [atn, encoder, decoder, stn, channel_sim]
    for model in models:
        model.train(is_train)

    rng = np.random.default_rng(SEED + epoch_index)
    total_loss = 0.0
    total_mse = 0.0
    total_samples = 0
    error_sum = 0.0
    power_sum = 0.0

    iterator = dataset.iterate_split(
        split,
        batch_size=cfg.batch_size,
        shuffle=is_train,
        generator=rng if is_train else None,
        fixed_snr=fixed_snr,
    )

    for batch_idx, (H_d, H_u, snr) in enumerate(iterator):
        if is_train and debug_train_batches is not None and batch_idx >= debug_train_batches:
            break
        if (not is_train) and debug_eval_batches is not None and batch_idx >= debug_eval_batches:
            break

        H_d = H_d.to(device, non_blocking=True)
        H_u = H_u.to(device, non_blocking=True)
        snr = snr.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            H_hat = forward_pass(H_d, H_u, snr)
            total_batch_loss, mse_loss, nmse_linear = compute_loss(H_d, H_hat, epoch_index)
            if is_train:
                total_batch_loss.backward()
                if cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(all_params, cfg.grad_clip)
                optimizer.step()

        batch_size = H_d.size(0)
        total_loss += float(total_batch_loss.detach().item()) * batch_size
        total_mse += float(mse_loss.item()) * batch_size
        total_samples += batch_size

        H_d_denorm = dataset.denormalize(H_d.detach(), "dl")
        H_hat_denorm = dataset.denormalize(H_hat.detach(), "dl")
        error_sum += float(torch.sum((H_d_denorm - H_hat_denorm) ** 2).item())
        power_sum += float(torch.sum(H_d_denorm ** 2).item())

    mean_loss = total_loss / max(total_samples, 1)
    mean_mse = total_mse / max(total_samples, 1)
    nmse_db = nmse_db_from_sums(error_sum, power_sum)

    return {
        "loss": mean_loss,
        "mse": mean_mse,
        "nmse_db": nmse_db,
        "linear_nmse": error_sum / max(power_sum, 1e-12),
        "samples": total_samples,
    }


def save_checkpoint(epoch, best_linear_nmse, tag):
    checkpoint = {
        "epoch": epoch,
        "cfg": cfg.__dict__,
        "stats": stats,
        "best_linear_nmse": best_linear_nmse,
        "atn_state_dict": atn.state_dict(),
        "encoder_state_dict": encoder.state_dict(),
        "decoder_state_dict": decoder.state_dict(),
        "stn_state_dict": stn.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
    }
    path = Path(cfg.checkpoint_dir) / tag
    torch.save(checkpoint, path)
    print(f"Saved checkpoint: {path}")


def find_latest_checkpoint():
    checkpoint_dir = Path(cfg.checkpoint_dir)
    candidates = list(checkpoint_dir.glob("epoch_*.pth"))
    candidates += [checkpoint_dir / "final_model.pth", checkpoint_dir / "best_model.pth"]
    candidates = [path for path in candidates if path.exists()]
    if not candidates:
        raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir}")
    return max(candidates, key=lambda path: path.stat().st_mtime)


def load_checkpoint(path):
    checkpoint = torch.load(path, map_location=device)
    atn.load_state_dict(checkpoint["atn_state_dict"])
    encoder.load_state_dict(checkpoint["encoder_state_dict"])
    decoder.load_state_dict(checkpoint["decoder_state_dict"])
    stn.load_state_dict(checkpoint["stn_state_dict"])
    if "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    print(f"Loaded checkpoint: {path}")
    return checkpoint


history = {
    "train_loss": [],
    "train_nmse_db": [],
    "val_loss": [],
    "val_nmse_db": [],
    "lr": [],
}


resume_state = None
if cfg.checkpoint_path:
    resume_state = load_checkpoint(cfg.checkpoint_path)
elif cfg.resume_latest:
    latest_checkpoint = find_latest_checkpoint()
    cfg.checkpoint_path = str(latest_checkpoint)
    resume_state = load_checkpoint(latest_checkpoint)


if cfg.run_training:
    start_epoch = 0
    best_val_linear_nmse = float("inf")
    if resume_state is not None:
        start_epoch = int(resume_state.get("epoch", -1)) + 1
        best_val_linear_nmse = float(resume_state.get("best_linear_nmse", float("inf")))
        print(
            f"Resuming training from epoch {start_epoch} "
            f"with best linear NMSE {best_val_linear_nmse:.6e}"
        )
        if start_epoch >= cfg.epochs:
            print(
                f"Checkpoint already reached epoch {start_epoch}. "
                f"Increase --epochs above {start_epoch} to continue training."
            )
    print(f"Starting training for {cfg.epochs} epochs")
    for epoch in range(start_epoch, cfg.epochs):
        start = time.time()
        train_metrics = run_epoch("train", epoch_index=epoch)
        val_metrics = run_epoch("val", epoch_index=epoch)

        scheduler.step(val_metrics["linear_nmse"])
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_metrics["loss"])
        history["train_nmse_db"].append(train_metrics["nmse_db"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_nmse_db"].append(val_metrics["nmse_db"])
        history["lr"].append(current_lr)

        elapsed = time.time() - start
        print(
            f"Epoch [{epoch + 1}/{cfg.epochs}] "
            f"| {elapsed:.1f}s "
            f"| LR {current_lr:.2e} "
            f"| Train Loss {train_metrics['loss']:.6f} "
            f"| Train NMSE {train_metrics['nmse_db']:.2f} dB "
            f"| Val Loss {val_metrics['loss']:.6f} "
            f"| Val NMSE {val_metrics['nmse_db']:.2f} dB"
        )

        if val_metrics["linear_nmse"] < best_val_linear_nmse:
            best_val_linear_nmse = val_metrics["linear_nmse"]
            save_checkpoint(epoch, best_val_linear_nmse, "best_model.pth")

        if (epoch + 1) % cfg.save_every == 0:
            save_checkpoint(epoch, best_val_linear_nmse, f"epoch_{epoch + 1}.pth")

    save_checkpoint(cfg.epochs - 1, best_val_linear_nmse, "final_model.pth")
else:
    print("Training is disabled. Set cfg.run_training = True and rerun the config cell to start training.")


In [ ]:
def evaluate_snr_sweep(snr_points, split="test"):
    results = []
    for snr_db in snr_points:
        metrics = run_epoch(split, fixed_snr=snr_db)
        results.append(metrics["nmse_db"])
        print(f"SNR {snr_db:>4} dB -> NMSE {metrics['nmse_db']:.3f} dB")
    return results


if cfg.run_evaluation:
    snr_points = [-10, -5, 0, 5, 10]
    nmse_values = evaluate_snr_sweep(snr_points, split="test")

    plt.figure(figsize=(8, 5))
    plt.plot(snr_points, nmse_values, marker="o", linewidth=2)
    plt.grid(True, alpha=0.3)
    plt.xlabel("SNR (dB)")
    plt.ylabel("NMSE (dB)")
    plt.title("NMSE vs SNR")

    if script_args is None:
        plt.show()
    else:
        plot_path = Path(cfg.checkpoint_dir) / "nmse_vs_snr.png"
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        print(f"Saved plot to {plot_path}")
else:
    print("Evaluation is disabled. For scripts use --evaluate or --train.")


## Notes

- `k` is the number of uplink subcarriers used for CSI feedback.
- `fast_dev_run=True` runs only a few batches per split and is useful for smoke testing.
- The tuned notebooks optimize a mixed MSE plus linear-NMSE loss after a short warmup.
